# Credit card fraud detection

# LightGBM | XGBoost Pipeline only!

## Introduction

The machine learning project aims to detect fraudulent credit card transaction, which is a growing problem for the banking industry and also for consumers. Due to growing scam and social engineering activities more people are victims. Therefore there are multiple stakeholders (regulatory authorities EBA & ECB, banks, consumers) interested in a sufficient fraud prevention and detection. 



The project is a classification problem, with highly unbalanced proportions of our to explainable variable y (is_fraud). Classifing the transaction into fraudulent transaction, while actually being fraudulent transaction is crucial for the credit card originating bank. Not detecting fraudulent transaction leads to financial liability damages, binding workers recourses and also damaging the reputation of the bank as safe financial service provider. Marked false positive transactions are also a problem for the bank business. Getting warning messages to the client or even blocking the credit card results into less card activity and therefore less revenue and profit for the bank. In the worst case clients will even terminate their credit card contract, because the credit card is not reliable when it should be. The declined using of a (business) credit card, for a late hotel check in, is horrible and for sure not a nice experience in a tight working schedule.    


This project considers the current developments in academic discussion and uses their approaches, mainly the recent published paper by "Shi et al (2025): An attention-based balanced variational autoencoder method for credit card fraud detection", on a different dataset which is publicly available on Hugging Face by the following link:

https://huggingface.co/datasets/pointe77/credit-card-transaction

The credit card data (most likely synthetic data) is downloaded from Hugging face, without providing any descriptions.Nevertheless the used variables are quit obvious, by their structure and their content. So the qualitative data description will be conducted on industrial experience and best guesses.


The Script CC_Fraud_DATA_Prep.ipynb already creates via feature engineering variables from the initial dataset which is based on the initial Hugging Face dataset and the FBI Crime Data. To make this script cleaner this script starts, with the used dataset description and then provides the model usage.  

## PROBLEM The engineered feature: dist_client_merchant was used to generate the variable is Fraud!

## General steps in this Project
As first step the dataset is changed and enriched with constructed variables (feature engineering) to get a better explanibility for classification fraud. Those steps are seperated in the script (CC_Fraud_Data_Prep.ipynb). A descriptive analysis of the constructed and original dataset is conducted, by using descriptive statistics and some visualisations. Since the dataset is not just a simple cross sectional dataset, not considering any time dependet relations could be fatally missing any crucial trends. A short time series analysis of our dataset is therefore included.

The feature enigineering of variables leads in multiple variables which are highly correlated to each other (by being based on the same features). Therefore the project contains LightGBM / Lasso Regression just for the purpose of deciding and selecting variables, which variables more efficiently leads to a better explanibilty. 
Both procedures are included, to get a more decisive analysis of the variable, whereas the lasso regression investigates linear relationship it provides negative or positive links to our fraud marker (y), which is a valuable information for the bank and future steps in avoiding fraud.  


In [ ]:
# Importing standard packages
import polars as pl # Faster then pandas, 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # visualization
#import sklearn as 

from geopy.distance import distance #, geodesic, great_circle # Feature Engineering module for Geodata

In [ ]:
# reading in the datasets

cc_train =  pl.read_parquet("credit_card_trans_train.parquet")
cc_test  = pl.read_parquet("credit_card_trans_test.parquet")
# cc_comb = pl.concat([cc_train, cc_test])
# del(cc_train, cc_test)
ID_cols = ['Owner_ID', 'cc_num', 'trans_num','trans_date_trans_time','merchant', 'dist_client_merchant']
cc_train_ID = cc_train.select(ID_cols)
cc_test_ID = cc_test.select(ID_cols)

cc_train = cc_train.drop(['Owner_ID', 'cc_num','trans_date_trans_time','merchant', 'dist_client_merchant'])
cc_test = cc_test.drop(['Owner_ID', 'cc_num','trans_date_trans_time','merchant', 'dist_client_merchant'])


In [ ]:
cc = cc_train
# Trans_num as ID and seperate Information DataFrame 
# To make the Data_Frame nicer to handle all Identifier variables and some string variables are seperated from the dataset:
#  
"""
cc_ID = 
print(f"Columns: {cc.columns.drop('')}")
"""
cc.head(5)


### A Short general description of the dataset and the difference to the original dataset.

The first variables contain Keyident data on different levels. We originated an unique OwnerID instead of the name, a CC_num (Credit Card Number) and trans_num (Transaction number) already existed, to identify fraudulent transaction and the related owner and card. 
The original dataset contains location data which is partly written as string dtype and also in latitude and longitude of customer and also merchant data. Since we derived in the feature engineering script the following variables: #LIST# -> The Variables: street	city	state	zip	lat	long merch_lat	merch_long merch_zipcode are deleted. 
The transaction date information is used to derive daily and weekly timestamps and based on these timestamps and location data the following features are derived: #LIST#.



Threshold in noon (2020, 6, 21)

## First Overview of the Data

### Qualitative Description of the DATA, What the Variables mean.



In [ ]:
# Since we got labeled data we can check the Time Series of fraud activity (at least for this data, the reality might be different with unlabeled data):
cc = cc_train
min_date = cc['trans_date'].min()

# weeks since start
cc = cc.with_columns(
    ((pl.col('trans_date') - min_date).dt.total_days() / 7).floor().cast(pl.Int32).alias('week_index')
)

# Aggregation weekly data
weekly_stats = cc.group_by(['week_index']).agg([
    pl.len().alias('total_transactions'),
    pl.col('is_fraud').sum().alias('fraud_count'),
    (pl.col('is_fraud').sum() / pl.len() * 100).alias('fraud_percentage')
]).sort(['week_index'])


In [ ]:
# Plot using week_index
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(weekly_stats['week_index'], weekly_stats['total_transactions'], 
         label='Total Transactions', color='blue', linewidth=2)
ax1.set_xlabel('Week Number (since start)')
ax1.set_ylabel('Total Transactions Count', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(weekly_stats['week_index'], weekly_stats['fraud_count'], 
         label='Fraud Count', color='red', linewidth=2, marker='o', markersize=4)
ax2.set_ylabel('Fraud Count', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Weekly Transactions vs Fraud Count')

cc.drop(['trans_date','week_index'])

Checking the Time Series visualy, the following seems to apply:
The fraud count seems to be stationary. Dickey Fuller Test can be conducted as addition.
The CC usage seems to get some seasonality, unfortunately the data does not cover multiple years (annualy seasonality seems to be dominating) and therefore it might be unsufficient to feature engineer. 
The rise of CC usage for christmas and then the low account balance afterwards is clearly visable. 

No strong trend (fraud isn't systematically increasing or decreasing)
No seasonality (no weekly, monthly, or quarterly patterns)
No memory effect (past fraud doesn't predict future fraud)

## ENCODING

II.1) Feature Representation
TODO: Adapt text to newer version
By using a ordinal range/categories for the variables,  it results in some better model performances. 
(Linear SVM, Logistic Regression) -Linear SVM for this data not used.
Whereas we will use the scaled integer values, for our tree based models. (XGBoost, LightGBM)  

The following Variables are therefore transformed:

- City Population
- State Risk Rating

Ordinal text variables:
Following the Lab4 - OrdinalEncoder will be used from the preprocessing module in Scikit-learn. 

In comparison to the algorithm of the Machine Learning course provided by UWM, in this tree based pipeline a more simplified preprocessing is conducted. First of all, since there are no missings in the dataset (Which is not totally unrealistic for those personal data and transaction data, considering the high regulation of data lineage and data quality in banks, their credit card transaction data is originated in their banking systems. It is more likelly that the personal data is not actual, which happens often, but is not easy to measure or consider.) the imputation step is skipped.
The scaling of numeric variables is for tree based alorithms skipped as well, since the splits are based on value comparisons.


In [ ]:
# cleaning up preprocessing and the pipeline:
"""
Dr. Example:

num_values_pipeline = make_pipeline(
    SimpleImputer(strategy='mean'),      # Imputing is not necessary in this dataset
    MinMaxScaler(),                      # Numeric variables does not need a scaling for TreeBased Algos
)
cat_values_pipeline = make_pipeline(
    OrdinalEncoder(handle_unknown='error'), # Encoding more to differiante
)

preprocessing_pipeline = ColumnTransformer([
    ('num_attributes_steps', num_values_pipeline, data.select_dtypes('number').columns),
    ('cat_attributes_steps', cat_values_pipeline, ('protocol_type', 'service', 'flag', 'labels')),
])

THEN AFTER preprocessing_pipeline:

data_preprocessed = pd.DataFrame(
    preprocessing_pipeline.fit_transform(data), 
    columns=preprocessing_pipeline.get_feature_names_out(),
    index=data.index,
)

"""


In [ ]:
' Checking Values Descriptive Stats'

(Dornadula & Geetha, 2019): "Firstly, we use clustering method to divide the cardholders into different clusters/groups based on their transaction amount, i.e., high, medium and low using range partitioning.  Using Sliding-Window method, we aggregate the transactions into respective groups, i.e., extract some features from window to find cardholder's behavioural patterns. Features like maximum amount, minimum amount of transaction, followed by the average amount in the window and even the time elapsed."
/* features extraction related to amount */  ai1=MAX_AMT(Ti);  ai2=MIN_AMT(Ti);  ai3=AVG_AMT(Ti);  ai4=AMT(Ti);  For j in range i+w-1:  /* Time elapse */  xi= Time(tj)-Time(tj-1)  End  Xi= (ai1, ai2,ai3,ai4,ai5,);

In [ ]:
def stat_values(df: pl.DataFrame, numeric_vars: list) -> pl.DataFrame:
    
    stats_list = []
    for col in numeric_vars: 
        stats = {
            "variable": col,
            "mean": df[col].mean(),
            "median": df[col].median(),
            "std": df[col].std(),
            "variance": df[col].var(),
            "min": df[col].min(),
            "q1": df[col].quantile(0.25),
            "q3": df[col].quantile(0.75),
            "max": df[col].max(),
            "range": df[col].max() - df[col].min(),
            "iqr": df[col].quantile(0.75) - df[col].quantile(0.25),
            "skewness": df[col].skew(),
            "kurtosis": df[col].kurtosis(),
        }
        stats_list.append(stats)
    
    return pl.DataFrame(stats_list)

# Usage
stats_df = stat_values(cc, ['amt','trans_time_diff','travel_time_km','daily_trans_lagged'])
stats_df

In [ ]:

# extreme outliers: log transform; net_binary_comb_travel_time, net_binary_comb_trans_time


Amounts got extreme Outliers. In Basic Data.
Transformation (No 0 or negative Values makes it easier) -> log transformation.


In [ ]:
def categorical_stats(df: pl.DataFrame, categorical_vars: list) -> pl.DataFrame:
    stats_list = []
    for col in categorical_vars:
        
        value_counts = df[col].value_counts()
                
        most_frequent = value_counts[0, col]
        most_freq_count = value_counts[0, "count"] 
        
        stats = {
            "variable": col,
            "n_unique": df[col].n_unique(),
            "most_frequent": most_frequent,
            "most_freq_pct": (most_freq_count / df.height) * 100,
        }
        stats_list.append(stats)
    
    return pl.DataFrame(stats_list)

cat_stats = categorical_stats(cc, ['category','State_Risk_Rating','Population_Density'])#,'generation'])

# None used variables Job

In [ ]:
cat_stats

In [ ]:
# There might be TWO 'amt' columns in your Polars DataFrame
print(cc.columns)  # Look for duplicate 'amt' columns

# If there's 'amt' and 'amt_right' or something similar

II.1) Feature Representation
TODO:
By using a ordinal range/categories for the variables,  it results in some better model performances. 
(Linear SVM, Logistic Regression) -Linear SVM for this data not used.
Whereas we will use the scaled integer values, for our tree based models. (XGBoost, LightGBM)  

The following Variables are therefore transformed:

- City Population
- State Risk Rating

Ordinal text variables:
Following the Lab4 - OrdinalEncoder will be used from the preprocessing module in Scikit-learn. 

## Preprocessing and Scaling different for the steps: LASSO - LSVM | LightGBM - XGBoost

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Preprocessing with encoding 

preprocessor = ColumnTransformer([
    ('state_risk', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['State_Risk_Rating']),
    ('population', OrdinalEncoder(categories=[['Village', 'Small_Town', 'Town', 'City', 'Metropolitan_Area']], handle_unknown='use_encoded_value', unknown_value=-1), ['Population_Density']),
    #('generation', OrdinalEncoder(categories=[['War_generation', 'Boomer', 'Gen X', 'Millennials', 'Gen Z']], handle_unknown='use_encoded_value', unknown_value=-1), ['generation']),
    ('pass', 'passthrough', [
        'amt', 'trans_time_diff', 
        'travel_time_km', 'daily_trans_lagged', 
        'net_binary',  'birth_year',
        'net_binary_comb_trans_time','net_binary_comb_travel_time'
    ]),  # unchanged variables
]) 

Maybe Winsorization for some variables (extreme outliers....), therefore cutting them...

## Light GBM as feature selection

To filtering useless features and also checking the hirarchy of our highly multicollinearated features, to make our final decision about the model input features, LightGBM is used. 
This approach follows the publication by S. Shi et al. (2025, p. 3): "We apply LightGBM as the tool to conduct an #automatic feature selection process.LightGBM, developed by Microsoft in 2017, is a tree-based gradient-boosting framework. Its most extraordinary advantages are efficiency and distributed computing. There are built-in functions that can calculate feature importance scores automatically and visualize the sortation of importance."

In [ ]:
import lightgbm as lgb
from sklearn.pipeline import Pipeline
# pipeline with LightGBM (Non Linear relationships)
pipeline_LGB = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        class_weight='balanced'
    ))
])

In [ ]:
X = cc[['amt',
 #,
 'trans_time_diff',
 'travel_time_km',
 'daily_trans_lagged',
 'net_binary_comb_travel_time',
 'net_binary_comb_trans_time',
 'birth_year',
 #'weekly_transactions',
 'net_binary',
# 'Amount_Outlier_bin',
 'State_Risk_Rating',
 'Population_Density']].to_pandas()

y = cc['is_fraud'].to_numpy()

In [ ]:
# After creating X, verify amount is still realistic
print("Amount in X (after selection):")
print(f"Min: {X['amt'].min():.2f}")
print(f"Max: {X['amt'].max():.2f}")
print(f"Mean: {X['amt'].mean():.2f}")
print(f"Unique values (first 10): {X['amt'].value_counts().head(10)}")

In [ ]:
pipeline_LGB.fit(X, y)

feature_importance = pipeline_LGB.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'feature': pipeline_LGB.named_steps['preprocessor'].get_feature_names_out(),
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(importance_df.head(20))

Tree based Multicollinearity is not an issue!

The result of LGBM leads to the following list of features: 
All amount variables are relevant, which is to PAPER CITATION!.
Daily_transactions was excluded as weekly_transactions provides a more stable and comprehensive measure of transaction frequency while capturing the same underlying pattern. Net_binary was retained over net_binary_comb_trans_time due to its stronger predictive signal. Birth_year was kept instead of the generation classification variable, because it showed higher importance.

In [ ]:
import pandas as pd
correlation_matrix = pd.DataFrame({
    'amt': X['amt'],
    #: X[], not allowed because global variable!
    #'Amount_Outlier_bin_comb_amt': X['Amount_Outlier_bin_comb_amt'], same problem
    'travel_time_km': X['travel_time_km'], # Out
    'net_binary_comb_travel_time': X['net_binary_comb_travel_time'], 
    'net_binary_comb_trans_time': X['net_binary_comb_trans_time'],
    'trans_time_diff': X['trans_time_diff'] # Out
}).corr()
correlation_matrix

## Selection of Features

In [ ]:
# Feature Update.
relevant_Feat=['amt',
 'daily_trans_lagged',
 #'dist_client_merchant', # ERROR perfect predictability
 #'trans_time_diff', OUT!
 'travel_time_km',
 #'daily_trans_lagged', OUT!
 'net_binary_comb_travel_time', # Error 
 'net_binary_comb_trans_time', # error predictability
 #'Amount_Outlier_bin_comb_amt',
 'birth_year',
 #'weekly_transactions',
 #'net_binary', OUT!
 #'Amount_Outlier_bin', OUT!
 'State_Risk_Rating',
 #'generation', OUT!
 'Population_Density']

Mentioned models for CC Fraud Detection in 

Dornadula, V. N., & Geetha, S. (2019). Credit Card Fraud Detection using Machine Learning Algorithms. Procedia Computer Science, 165, 631–641. https://doi.org/10.1016/j.procs.2020.01.057

XGBoost, KNN, SVM.

Decision to use XGBoost & LinearSVM (instead of KNN), because of more dimensionalities and also faster calculation.

Also in:
(Tiwari et al., 2021, p. 4):"Support Vector Machines  Support vector machines or SVMs are linear classifiers as stated in [5] that work in high dimensionality because in high-dimensions, a non-linear task in input becomes linear and hence this makes SVMs highly useful for detecting frauds. Due to its two most important features that is a kernel function to represent classification function in the dot product of input data point projection, and the fact that it tries finding a hyperplane to maximize separation between classes while minimizing overfitting of training data, it provides a very high generalization capability."

In [ ]:
print("Columns in X:", X.columns.tolist())
print("\nIs 'is_fraud' in X?", 'is_fraud' in X.columns)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform

In [ ]:
# Update in the preprocessor:
relevant_feat_pass = relevant_Feat.copy()
relevant_feat_pass.remove('State_Risk_Rating')
relevant_feat_pass.remove('Population_Density')

preprocessor = ColumnTransformer([
    ('state_risk', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['State_Risk_Rating']),
    ('population', OrdinalEncoder(categories=[['Village', 'Small_Town', 'Town', 'City', 'Metropolitan_Area']], handle_unknown='use_encoded_value', unknown_value=-1), ['Population_Density']),
    #('generation', OrdinalEncoder(categories=[['War_generation', 'Boomer', 'Gen X', 'Millennials', 'Gen Z']], handle_unknown='use_encoded_value', unknown_value=-1), ['generation']),
    ('pass', 'passthrough',relevant_feat_pass),  # unchanged variables
]) 

In [ ]:
## XGBoost as Model
#preprocessor = preprocessor.transformers.pop(2) #dropping generation encoding

pipeline_xgboost = Pipeline([
    ('preprocessor', preprocessor), 
    ('classifier', xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ))
])

In [ ]:
pipeline_xgboost

In [ ]:
hiperparam_distributions = {
    'classifier__n_estimators': randint(50, 100),           # Number of trees
    'classifier__max_depth': randint(3, 10),                # Tree depth - pretty high with 8
    'classifier__learning_rate': uniform(0.01, 0.3),        # Step size shrinkage - learning rate seems about 5% reasonable
    'classifier__subsample': uniform(0.6, 0.4),             # Row sampling
    'classifier__colsample_bytree': uniform(0.6, 0.4),      # Column sampling
    'classifier__min_child_weight': randint(1, 10),         # Min instances in leaf
    'classifier__gamma': uniform(0, 0.5),                   # Min loss reduction
    'classifier__reg_alpha': uniform(0, 1),                 # L1 regularization
    'classifier__reg_lambda': uniform(0.5, 1.5)             # L2 regularization
}

In [ ]:
randomized_search_xgb = RandomizedSearchCV(
    pipeline_xgboost,
    hiperparam_distributions,
    n_iter=10,
    cv=5,
    scoring={'roc_auc': 'roc_auc',
        'accuracy': 'accuracy'},
    refit='roc_auc',                 
    random_state=42,
    n_jobs=-1,                         
    verbose=1 
)


In [ ]:
x = X[relevant_Feat]
x.head(5)

In [ ]:
randomized_search_xgb.fit(x, y)
print("Best params:", randomized_search_xgb.best_params_)
print("Best score:", randomized_search_xgb.best_score_)

In [ ]:
from xgboost import XGBClassifier
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),  
    ('classifier', XGBClassifier(
        n_estimators = randomized_search_xgb.best_params_['classifier__n_estimators'],
        max_depth = randomized_search_xgb.best_params_['classifier__max_depth'],
        learning_rate = randomized_search_xgb.best_params_['classifier__learning_rate'],
        subsample = randomized_search_xgb.best_params_['classifier__subsample'],
        colsample_bytree = randomized_search_xgb.best_params_['classifier__colsample_bytree'],
        min_child_weight = randomized_search_xgb.best_params_['classifier__min_child_weight'],
        gamma = randomized_search_xgb.best_params_['classifier__gamma'],
        reg_alpha  = randomized_search_xgb.best_params_['classifier__reg_alpha'],
        reg_lambda = randomized_search_xgb.best_params_['classifier__reg_lambda'],
        random_state=42,  
        eval_metric='logloss'  
    ))
])

In [ ]:
X_test = cc_test[relevant_Feat].to_pandas()  
y_test = cc_test['is_fraud'].to_pandas()
X_test

In [ ]:
# Predict on test set
best_pipeline.fit(x, y) 
y_pred_test = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_class = best_pipeline.predict(X_test)

# Calculate performance
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

auc_test = roc_auc_score(y_test, y_pred_test)
accuracy_test = accuracy_score(y_test, y_pred_class)

In [ ]:
for col in X_test.columns:
    if col != 'dist_client_merchant':  # Already found
        fraud_vals = set(X_test[col][y_test == 1].unique())
        nonfraud_vals = set(X_test[col][y_test == 0].unique())
        
        if len(fraud_vals) > 0 and len(nonfraud_vals) > 0:
            overlap = fraud_vals.intersection(nonfraud_vals)
            if len(overlap) == 0:
                print(f"⚠️ ALSO PERFECT: {col}")

In [ ]:
# Test each feature individually for predictive power
from sklearn.tree import DecisionTreeClassifier

print("Individual feature predictive power (AUC):")
print("="*50)

for col in ['amt', 'trans_time_diff', 'travel_time_km', 'daily_trans_lagged', 'birth_year']:
    if col in x.columns:
        model = DecisionTreeClassifier(max_depth=3, random_state=42)
        model.fit(x[[col]], y)
        y_pred = model.predict_proba(X_test[[col]])[:, 1]
        auc = roc_auc_score(y_test, y_pred)
        print(f"{col:<25} AUC: {auc:.4f}")

# Check if ANY single feature gives >0.85 AUC
print("\n" + "="*50)
print("If any single feature has AUC > 0.85, your data is artificially separated")

In [ ]:
auc_test, accuracy_test

In [ ]:
fraud_rate = y_test.mean()
print(f"Fraud rate in test: {fraud_rate:.4f}")

In [ ]:
def plot_parameter_impact_simple(search_results):
    results = pd.DataFrame(search_results.cv_results_)
    
    # Extract parameters
    param_cols = [col for col in results.columns if col.startswith('param_classifier__')]
    
    for param in param_cols:
        param_name = param.replace('param_classifier__', '')
        param_means = results.groupby(param)['mean_test_score'].mean()
        
        plt.figure(figsize=(8, 4))
        plt.plot(param_means.index.astype(str), param_means.values, 'o-', linewidth=2, color='blue')
        plt.xlabel(param_name)
        plt.ylabel('Mean CV Score (AUC)')
        plt.title(f'Impact of {param_name}')
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# This will show each parameter in a separate plot
plot_parameter_impact_simple(randomized_search_xgb)

In [ ]:
# Show top 5 parameter combinations
results = pd.DataFrame(randomized_search_xgb.cv_results_)
top5 = results.nlargest(5, 'mean_test_score')[['params', 'mean_test_score', 'std_test_score']]
top5

In [ ]:
best_xgboost = randomized_search_xgb.best_estimator_

Concept of Voting Classifier.
Lasso Regression, LightGBM -> Feature Selection
Since the  XGBoost, LSVC as Model -> Voting if 


# Analyis

In qualitative variable analysis, the
- construction of our Amount Outlier might be discussed. (Values 1.5, 2, 2.5 etc * IQR in threshold)  
- City Population might be differently calculated, by choosing different ranges or more/less categories...
- generational variable

Set Up Discussions:
But a feature could be useful in LightGBM but useless in XGBoost (and vice versa) because:
Different splitting criteria (gain vs. gini vs. mse)
Different handling of categorical features
Different tree growth strategies (leaf-wise vs. depth-wise)